In [16]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.datasets import make_regression
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV

Mając zbiór danych z wieloma cechami (część nieistotnych, dodanych jako szum), napisz funkcję compare_regularization(X, y), która trenuje LinearRegression, Ridge i Lasso, porównuje ich błędy testowe oraz pokazuje, które współczynniki Lasso wyzerował.

Lasso i Ridge - obie metody służą do regularyzacji, czyli zapobiegania przeuczeniu (overfittingowi) poprzez nakładanie "kary" na model za posiadanie zbyt dużych wag.

In [17]:
# n informative 2 oznacza ze 2 cechy sa 

In [18]:
X, y = make_regression(n_samples=200, n_features=4, noise=1, n_informative=2, random_state=42)

In [19]:
#wzielismy 20 procent jako test czyli 20 procent danych zamknelismy i na 80 bedziemy trenowac model 
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state =42)
X

array([[ 0.45918008,  0.83392215,  0.55979045,  1.08078073],
       [-0.13014305,  0.32416635,  0.68195297, -0.31026676],
       [-1.19620662, -1.10633497, -0.47917424, -0.18565898],
       [-2.30192116, -0.2750517 , -0.53050115, -0.57581824],
       [-0.12578692, -0.98960482,  1.59318663, -0.51121568],
       [ 0.06856297, -1.55066343,  0.09965137, -0.50347565],
       [-0.54342477, -0.03275327, -0.57366201, -0.54685894],
       [-0.48423407, -1.51936997,  2.1221562 ,  1.03246526],
       [-0.59157139,  1.2776649 , -0.02090159,  0.11732738],
       [ 0.17457781,  1.8861859 , -0.16128571,  0.40405086],
       [ 0.70030988, -0.85835778,  0.4933179 ,  0.18483612],
       [ 0.75193303,  1.14282281, -0.03471177, -1.16867804],
       [-0.38508228,  0.32408397,  0.34361829, -1.76304016],
       [ 0.20346364, -1.60644632, -0.12791759, -0.95554044],
       [ 0.6206721 , -0.07443343, -0.72713718, -0.24751864],
       [ 0.53891004,  1.52312408,  0.07736831, -0.8612842 ],
       [ 1.12656503,  1.

wbijamy na poczatek sama linerregresion czyli model matematyczny pusty
potem trenujemy go na danych train X_train i y train czyli na pytaniach testowych i odpowiedziach 

potem robimy model ridge w podobny sposob
Jak to działa pod maską (np. dla cv=5):
- Automat bierze Twój zbiór treningowy (X_train) i dzieli go w locie na 5 równych, mniejszych kawałków (foldów).
- Trenuje model na 4 kawałkach, a sprawdza na 1.
- Powtarza ten proces dla kilkudziesięciu różnych wartości alpha (domyślnie od bardzo małych do bardzo dużych).
- Wyciąga średnią z błędów i sam wybiera matematycznie idealną wartość kary, zanim w ogóle odda Ci gotowy model.

In [33]:
def compare_regularization(X,y):
    X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state =42)
    model_lr = LinearRegression()
    model_lr.fit(X_train,y_train)
    
    #X_train + y_train: To materiały do nauki. Model się uczył.
    #X_test: To arkusz egzaminacyjny. Model go widzi pierwszy raz. predict(X_test) to moment, w którym model wpisuje odpowiedzi na kartce.
    #y_test: To Twój oficjalny klucz odpowiedzi.
    #mean_squared_error: To porównanie odpowiedzi modelu z kluczem i wystawienie oceny.
    
    pred_lr = model_lr.predict(X_test)
    mse_lr = mean_squared_error(y_test, pred_lr)

    
    model_ridge = RidgeCV(cv=5)
    model_ridge.fit(X_train,y_train)
    pred_ridge = model_ridge.predict(X_test)
    mse_ridge = mean_squared_error(y_test, pred_ridge)

    model_lasso = LassoCV(cv=5)
    model_lasso.fit(X_train,y_train)
    pred_lasso = model_lasso.predict(X_test)
    mse_lasso = mean_squared_error(y_test, pred_lasso)

    return f" ridge mse: {mse_ridge}, lr mse: {mse_lr}, lasso mse: {mse_lasso} Wagi Lasso: {model_lasso.coef_}"

compare_regularization(X,y)

#zwracajac X_train pokazuje sie nam 80 procent z probki macierzy naszych dancych


' ridge mse: 1.277759666164882, lr mse: 1.2679498631498702, lasso mse: 1.7492758312159467 Wagi Lasso: [-0.          0.         31.35871202 82.48035048]'

 lasso wyeliminowal dwie pierwsze kolumny i uznal jako szum 

LR: Najniższe MSE, ale wysokie ryzyko przeuczenia, bo "widzi" sygnał tam, gdzie go nie ma (w szumie).

Lasso: Nieco wyższe MSE, ale najwyższa jakość modelu. Model Lasso jest "czysty" – ignoruje szum i polega tylko na tym, co naprawdę ma znaczenie. W inżynierii wybierasz Lasso, bo budujesz system stabilny i odporny na zmienne warunki rynkowe.




 
 
 
 model ridge W praktyce oznacza to, że Ridge matematycznie 'spłaszcza' duże wagi, dociskając je w stronę zera, ale w przeciwieństwie do Lasso, prawie nigdy nie zeruje ich całkowicie. Używam tej metody głównie wtedy, gdy w danych występuje zjawisko współliniowości (czyli wiele cech jest ze sobą silnie skorelowanych)

Podsumowanie działania algorytmów:

Linear Regression: Naiwnie dopasował się do wszystkich zmiennych. Przypisał wagi również kolumnom z szumem, co na realnym rynku prowadzi do przeuczenia (overfittingu).

Ridge (L2): Zastosował zarządzanie ryzykiem. Skompresował wagi wszystkich cech blisko zera, aby zapobiec skrajnościom, ale nie usunął żadnej z nich.

Lasso (L1): Przeprowadził twardą selekcję cech (Feature Selection). Wykrył brak korelacji w nieistotnych danych i wyzerował ich wagi do 0.0, zostawiając w modelu wyłącznie prawdziwy sygnał.

In [34]:
OPIS ZADANIA
"""
PODSUMOWANIE PROCESU TRENINGU I REGULARYZACJI:

1. PRZYGOTOWANIE DANYCH (make_regression):
   - Wygenerowano 200 próbek z 4 cechami (features).
   - 2 cechy: sygnał (wpływ na wynik), 2 cechy: szum (brak wpływu).
   - Podział (train_test_split): 80% do nauki, 20% do testu (egzamin końcowy).

2. DZIAŁANIE MODELI:
   - Linear Regression: "Chciwy" model. Wykorzystał wszystkie 4 kolumny. 
     Dopasował się do szumu, co skutkuje najniższym MSE, ale wysokim ryzykiem przeuczenia (overfitting).
   - Ridge (L2): Wygładził wagi wszystkich cech. Szum pozostał w modelu, ale z mniejszym wpływem.
   - Lasso (L1): "Inteligentny" selekcjoner. Zidentyfikował szum i wyzerował jego wagi (wartość 0.0). 
     Model został "oczyszczony", co zwiększa jego stabilność w przyszłych predykcjach.

3. INTERPRETACJA MSE (Mean Squared Error):
   - MSE mierzy błąd predykcji modelu na danych, których nie widział (X_test).
   - Niższe MSE (np. w LR) sugeruje lepsze dopasowanie historyczne, ale może być efektem "zakuwania szumu na pamięć".
   - Wyższe MSE w Lasso to cena za generalizację. Lasso świadomie ignoruje szum, dzięki czemu model jest bardziej odporny na zmienne warunki rynkowe (nie "ściąga" z danych treningowych).
"""

SyntaxError: invalid syntax (313746566.py, line 1)